# Project 2: Earnings Surprises and Stock Market Reactions

<div style="background:#e8f1fb; border-left:6px solid #2b6cb0; padding:8px 14px"><b>Goal.</b> When a company in SIC 38 (Medical Instruments) announces better-than-expected earnings, does its stock earn a higher return around the announcement?<br>
<b>Name:</b> Bea Macalalag &nbsp;|&nbsp; <b>Industry:</b> SIC 38, Medical Instruments &nbsp;|&nbsp; <b>Client:</b> Stryker (SYK)</div>

**The 6 steps**

| Step | What we do | Table we use | How tables are combined |
|---|---|---|---|
| 1 | Read the link file | `ccm_link.csv` | none |
| 2 | Pull quarterly EPS, price, and announcement date | `comp.fundq` | none |
| 3 | Add the CRSP id, then EPS from four quarters before, last quarter's price, SUE, and Rules 1-5 | funda + link | 2 merges |
| 4 | Pull daily returns and trading-day number | `crsp.dsf`, `crsp.dsi` | 1 merge |
| 5 | Find day 0 and calculated 3-day CAR | Steps 3 and 4 | 1 merge |
| 6 | Apply the rules and make the sample construction table | Step 5 | none |

**The measures**

| Measure | Formula |
|---|---|
| SUE (earnings surprise) | (EPS this quarter − EPS four quarters before) ÷ price at the end of last quarter |
| Abnormal return on one day | stock return − market return (value-weighted) |
| CAR (3-day abnormal return) | sum of the daily abnormal returns on days −1, 0 and +1, where day 0 is the first trading day on or after the announcement date |

**Four ways to join two tables** (you will choose one at every merge)

| Join | Rows that are kept |
|---|---|
| `inner` | only keys found in **both** tables |
| `left` | **all** rows of the left table; the right table's columns are empty where there is no match |
| `right` | all rows of the right table |
| `outer` (full) | all rows of both tables |

<h2 style="background:#1f3a5f; color:white; padding:8px 14px; border-radius:4px">Run this first. Session verification</h2>

In [1]:
# ===== Session verification (class notebook, unchanged) =====
import subprocess
from datetime import datetime

print("=" * 40)
print("SESSION VERIFICATION")
print("=" * 40)
print("Run time:", datetime.now())
print("Colab session ID:", subprocess.getoutput("hostname"))
print("=" * 40)

student_name = input("Enter your full name: ")
print("\nSubmitted by:", student_name)

SESSION VERIFICATION
Run time: 2026-11-21 12:16:02.896423
Colab session ID: wrds-hnt-01-w.wharton.private


Enter your full name:  Bea Macalalag



Submitted by: Bea Macalalag


<h2 style="background:#1f3a5f; color:white; padding:8px 14px; border-radius:4px">Setup. Libraries and settings</h2>

In [2]:
# ===== Setup: libraries and settings (class notebook; AI, see Appendix A; changed the industry, dates and rules) =====
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)   # hide a harmless pandas warning

import pandas as pd
import wrds

conn = wrds.Connection()    # log in to WRDS (type your username and password if asked)

# Settings. If you want to change something, change it here only.

SIC        = "38"             # Medical Instruments 
ANN_START  = "2021-01-01"     # we study announcements (rdq) between ANN_START and ANN_END
ANN_END    = "2024-12-31"
PULL_START = "2019-01-01"     # we pull earlier quarters too, because SUE needs the EPS of four quarters before
RET_START  = "2020-12-01"     # first day of returns (covers day -1 or a January 2021 announcement)
RET_END    = "2025-01-31"     # last day of returns (covers day +1 of a December 2024 announcement)
MIN_PRC    = 5                # rule: drop a company-quarter if last quarter's price is below this
MAX_SUE    = 1                # rule: drop a company-quarter if SUE is bigger than this (up or down) 

Loading library list...
Done


<h2 style="background:#1f3a5f; color:white; padding:8px 14px; border-radius:4px">Step 1. Link file (ccm_link.csv)</h2>

<div style="background:#e8f1fb; border-left:6px solid #2b6cb0; padding:8px 14px"><b>Why we need it.</b> Compustat identifies a company with <code>gvkey</code>, CRSP identifies it with <code>permno</code>.
The link file tells us which <code>permno</code> belongs to which <code>gvkey</code>, and <b>between which dates</b> the link is valid.</div>

| Column | Meaning |
|---|---|
| `gvkey` | Compustat company id |
| `permno` | CRSP stock id |
| `linkdt` | first date the link is valid |
| `linkenddt` | last date the link is valid (empty = still valid) |
| `sic2` | first 2 digits of the industry code |

One `gvkey` can have **more than one** `permno` over time, so the dates matter when we merge in Step 3.

In [3]:
# ===== Step 1. Read the link file (class notebook, unchanged) =====

# 1a. Read every column as text, so that nothing is changed or guessed at this point.
link_all = pd.read_csv("ccm_link.csv", dtype=str)

# 1b. gvkey must have 6 characters (for example 001690). Excel can drop the leading zeros, so we put them back.
link_all["gvkey"] = link_all["gvkey"].str.zfill(6)

# 1c. permno is a number.
link_all["permno"] = pd.to_numeric(link_all["permno"]).astype(int)

# 1d. The two date columns are text. Turn them into real dates, so that we can compare them with other dates.
# format="mixed" accepts different date formats. errors="coerce" turns a bad value into a missing date.
link_all["linkdt"]    = pd.to_datetime(link_all["linkdt"], format="mixed", errors="coerce")
link_all["linkenddt"] = pd.to_datetime(link_all["linkenddt"], format="mixed", errors="coerce")

# 1e. A link with no end date is still valid, so we give it a far-future end date.
link_all["linkenddt"] = link_all["linkenddt"].fillna(pd.Timestamp("2099-12-31"))

# 1f. Keep only the companies in our industry.
link = link_all[link_all["sic2"] == SIC].reset_index(drop=True)
if len(link) == 0:
    raise ValueError("SIC " + SIC + " is not in ccm_link.csv. Industries in the file: " + str(sorted(link_all["sic2"].unique())))

print("link rows:", len(link), "| companies:", link["gvkey"].nunique())
link.head()

<h2 style="background:#1f3a5f; color:white; padding:8px 14px; border-radius:4px">Step 2. Compustat quarterly data (comp.fundq)</h2>

| Column | Meaning |
|---|---|
| `gvkey` | company id |
| `datadate` | last day of the fiscal quarter |
| `fyearq` | fiscal year |
| `fqtr` | fiscal quarter (1 to 4) |
| `rdq` | the day the company announced the quarter's earnings |
| `epspxq` | earnings per share (EPS) for the quarter, before extraordinary items |
| `prccq` | share price at the end of the fiscal quarter |

The four `indfmt / datafmt / popsrc / consol` lines in the query are the standard Compustat filters. They give **one row per company and fiscal quarter**.

In [4]:
# ===== Step 2. Pull the quarterly data (class notebook; AI, see Appendix A; changed to quarterly) =====

# 2a. The SQL query needs the gvkeys as one text:  '001690','002176',...
gv = "'" + "','".join(link["gvkey"].unique()) + "'"

# 2b. Ask WRDS for only the rows and columns we need.
fq = conn.raw_sql(f"""
    select gvkey, datadate, fyearq, fqtr, rdq, epspxq, prccq
    from comp.fundq
    where gvkey in ({gv})
      and datadate between '{PULL_START}' and '{ANN_END}'
      and indfmt = 'INDL' and datafmt = 'STD' and popsrc = 'D' and consol = 'C'
""", date_cols=["datadate", "rdq"])

# 2c. WRDS can return numbers in a special type. Turn them into plain decimal numbers (a missing value becomes NaN).
fq["epspxq"] = fq["epspxq"].astype("float64")
fq["prccq"]  = fq["prccq"].astype("float64")

# 2d. A row without a fiscal year or quarter cannot be placed in time, so we drop it.
fq = fq.dropna(subset=["fyearq", "fqtr"])

# 2e. Check what we got.
print("company-quarters pulled from Compustat:", len(fq))
fq.head()

<h2 style="background:#1f3a5f; color:white; padding:8px 14px; border-radius:4px">Step 3. Add the link, then last year's values, then the EPS change</h2>

<div style="background:#e8f1fb; border-left:6px solid #2b6cb0; padding:8px 14px"><b>Two merges in this step.</b><br>
<b>Merge 1.</b> <code>fundq</code> + <code>link</code> on <code>gvkey</code>, then keep only the link that was valid on the quarter-end date.<br>
<b>Merge 2.</b> The table + two copies of itself, moved forward in time, so that each row gets <b>the EPS of four quarters before</b> and <b>last quarter's price</b>.</div>

In [5]:
# ===== Step 3a. Merge 1: add permno from the link file (class notebook, unchanged) =====
panel = fq.merge(link[["gvkey", "permno", "ticker", "linkdt", "linkenddt"]], on="gvkey", how="inner")
print("3a. after the merge (a gvkey with 2 links is now on 2 rows):", len(panel))

3a. after the merge (a gvkey with 2 links is now on 2 rows): 4434


In [6]:
# ===== Step 3b. Keep only the link that was valid on the quarter-end date (class notebook; AI, see Appendix A; changed the key) =====

# True/False for each row: the date is between the start and the end of the link.
valid = (panel["linkdt"] <= panel["datadate"]) & (panel["datadate"] <= panel["linkenddt"])
panel = panel[valid]

# How many rows share the same gvkey + fyearq + fqtr with another row?
n_dup = panel.duplicated(["gvkey", "fyearq", "fqtr"]).sum()
print("3b. rows that repeat a gvkey + fyearq + fqtr before we drop them:", n_dup)

# If one company-quarter still has two rows, keep one.
panel = panel.drop_duplicates(["gvkey", "fyearq", "fqtr"])

# We do not need the link dates any more.
panel = panel.drop(columns=["linkdt", "linkenddt"]).reset_index(drop=True)

# Count (used in the sample construction table in Step 6).
r1, c1 = len(fq), fq["gvkey"].nunique()          # before the link
r2, c2 = len(panel), panel["gvkey"].nunique()    # after the link
print("3b. company-quarters after the link merge:", r2)

3b. rows that repeat a gvkey + fyearq + fqtr before we drop them: 4
3b. company-quarters after the link merge: 3650


**Duplicate check.** The key of this table is `gvkey` + `fyear`+ `fqtr`: **one row = one company and one fiscal year and one fiscal quarter**. Every key must appear **once**.
If the same key appears twice, a merge went wrong (for example, a company with two links and no date rule), and every later number is wrong.

In [7]:
# ===== Check: does any key (gvkey + fyear + fqtr) appear more than once? (class notebook; AI, see Appendix A; changed the key) =====
# duplicated() marks a row True if an earlier row has the same key. sum() counts the True values.
print("duplicate keys:", panel.duplicated(["gvkey", "fyearq", "fqtr"]).sum(), "(it must be 0)")

duplicate keys: 0 (it must be 0)


**Earlier quarters' values.** First we give every quarter one number: **quarter number = fiscal year × 4 + fiscal quarter**.
Then "four quarters ago" is the quarter number − 4, and "last quarter" is the quarter number − 1, even across a year.

| `fyearq` | `fqtr` | `qnum` |
|---|---|---|
| 2021 | 1 | 8085 |
| 2021 | 4 | 8088 |
| 2022 | 1 | 8089 |

We make **two copies** of the table and move them forward: copy 1 by **four** quarters (for EPS), copy 2 by **one** quarter (for price).
When we merge them back on `gvkey` + `qnum`, the row for 2022 Q1 meets the EPS of 2021 Q1 and the price of 2021 Q4.
A quarter with no earlier quarter finds no match, so its values are **empty (NaN)**. The merge must keep it (`how="left"`), and Rule 3 removes it later.

In [8]:
# ===== Step 3c. Merge 2: add EPS from four quarters ago and last quarter's price (class notebook; AI, see Appendix A; changed to quarterly) =====

# Quarter number: one number for every quarter. 2022 Q1 = 2022 * 4 + 1 = 8089.
panel["qnum"] = panel["fyearq"] * 4 + panel["fqtr"]

# Copy 1: move every EPS forward by four quarters. The EPS of 2021 Q1 is now on the row of 2022 Q1.
lag4 = panel[["gvkey", "qnum", "epspxq"]].copy()
lag4["qnum"] = lag4["qnum"] + 4
lag4 = lag4.rename(columns={"epspxq": "eps_lag4"})

# Copy 2: move every price forward by one quarter. The price of 2021 Q4 is now on the row of 2022 Q1.
lag1 = panel[["gvkey", "qnum", "prccq"]].copy()
lag1["qnum"] = lag1["qnum"] + 1
lag1 = lag1.rename(columns={"prccq": "prc_lag1"})

# Merge. how="left": keep every row of panel, even when there is no earlier quarter.
panel = panel.merge(lag4, on=["gvkey", "qnum"], how="left")
panel = panel.merge(lag1, on=["gvkey", "qnum"], how="left")
r3, c3 = len(panel), panel["gvkey"].nunique()

# SUE: (EPS this quarter - EPS four quarters ago) / last quarter's price.
panel["sue"] = (panel["epspxq"] - panel["eps_lag4"]) / panel["prc_lag1"]

panel[["gvkey", "fyearq", "fqtr", "epspxq", "eps_lag4", "prc_lag1", "sue"]].head(8)

In [9]:
# ===== Step 3d. Rules 1 to 3: keep the quarters with an announcement date in the sample period and a SUE (class notebook; AI, see Appendix A; changed to quarterly) =====
# A missing value does not pass a rule, so that quarter is removed.

# Rule 1: the announcement date (rdq) is known.
events = panel[panel["rdq"].notna()]
r4, c4 = len(events), events["gvkey"].nunique()

# Rule 2: the announcement is between ANN_START and ANN_END.
events = events[(events["rdq"] >= ANN_START) & (events["rdq"] <= ANN_END)]
r5, c5 = len(events), events["gvkey"].nunique()

# Rule 3: SUE can be computed.
events = events[events["sue"].notna()].reset_index(drop=True)
r6, c6 = len(events), events["gvkey"].nunique()

print("Rule 1, rdq is known:", r4)
print("Rule 2, rdq in the sample period:", r5)
print("Rule 3, SUE can be computed:", r6)

Rule 1, rdq is known: 3624
Rule 2, rdq in the sample period: 2412
Rule 3, SUE can be computed: 2333


In [10]:
# ===== Step 6b. Rule 4: last quarter's price is at least MIN_PRC (class notebook; AI, see Appendix A; changed to quarterly) =====
# A missing price is NOT >= 5, so those rows are removed too.
events = events[events["prc_lag1"] >= MIN_PRC]
r9, c9 = len(events), events["gvkey"].nunique()
print("announcements left:", r9)

announcements left: 1777


In [11]:
# ===== Step 6c. Rule 5: SUE is at most MAX_SUE, up or down (class notebook; AI, see Appendix A; changed to quarterly) =====
# abs() removes the minus sign, so -1.5 and +1.5 are both too big.
events = events[events["sue"].abs() <= MAX_SUE].reset_index(drop=True)
r10, c10 = len(events), events["gvkey"].nunique()
print("announcements left:", r10)

announcements left: 1775


<h2 style="background:#1f3a5f; color:white; padding:8px 14px; border-radius:4px">Step 4. Daily stock and market returns (crsp.dsf, crsp.dsi)</h2>

| Table | Column | Meaning |
|---|---|---|
| `crsp.dsf` | `permno` | stock id |
| | `date` | trading day |
| | `ret` | the stock's return on that day |
| `crsp.dsi` | `date` | trading day |
| | `vwretd` | the market return on that day (value-weighted) |

**Abnormal return on one day = stock return − market return.**
Daily dates of a stock and of the market are the same, so we merge on `date`.
We also number the trading days 1, 2, 3, … (`tday`). Weekends and holidays are not in the table, so the day before day 0 is always `tday − 1`.

In [12]:
# ===== Step 4. Pull the daily returns and compute the daily abnormal return (class notebook; AI, see Appendix A; changed to daily) =====

# 4a. The stocks we need: the permnos of the quarters in `events`.
permnos = ",".join(events["permno"].astype(str).unique())

# 4b. Ask WRDS for the daily stock returns and the daily market returns.
stock = conn.raw_sql(f"""select permno, date, ret from crsp.dsf
                         where permno in ({permnos}) and date between '{RET_START}' and '{RET_END}'""",
                     date_cols=["date"])
mkt = conn.raw_sql(f"""select date, vwretd from crsp.dsi
                       where date between '{RET_START}' and '{RET_END}'""",
                   date_cols=["date"])

# 4c. Plain numbers (a missing return becomes NaN).
stock["permno"] = stock["permno"].astype(int)
stock["ret"]    = stock["ret"].astype("float64")
mkt["vwretd"]   = mkt["vwretd"].astype("float64")

# 4d. Trading-day number: sort the market days by date and number them 1, 2, 3, ...
mkt = mkt.sort_values("date").reset_index(drop=True)
mkt["tday"] = mkt.index + 1

# 4e. Merge 3: add the market return and the trading-day number of the same date to every stock-day.
print("stock-days before Merge 3:", len(stock))
stock = stock.merge(mkt[["date", "vwretd", "tday"]], on="date", how="left")

# 4f. Abnormal return = stock return - market return.
stock["ar"] = stock["ret"] - stock["vwretd"]

print("stock-days after Merge 3:", len(stock), "| days with a missing return:", int(stock["ar"].isna().sum()))
stock.head()

<h2 style="background:#1f3a5f; color:white; padding:8px 14px; border-radius:4px">Step 5. Day 0 and the 3-day CAR of each announcement</h2>

<div style="background:#fdecea; border-left:6px solid #c53030; padding:8px 14px"><b>Watch out.</b> CAR is the <b>sum</b> of 3 daily numbers. If a day has no return
(the data is missing), the sum <b>skips it without any warning</b>, so a total of only 2 days looks like a normal number.
The protection is to <b>count the days</b> next to the sum (<code>days_found</code>).</div>

In [13]:
# ===== Step 5. Find day 0 and add up the 3 daily abnormal returns (class notebook; AI, see Appendix A; changed to daily) =====

# 5a. Give every announcement a number (event_id). merge_asof in 5b needs the dates in order, so we sort by rdq first.
events = events.sort_values("rdq").reset_index(drop=True)
events["event_id"] = events.index

# 5b. Day 0 = the first trading day on or after rdq. direction="forward" moves a Saturday announcement to Monday.
calendar = mkt[["date", "tday"]].rename(columns={"date": "day0", "tday": "n"})
events = pd.merge_asof(events, calendar, left_on="rdq", right_on="day0", direction="forward")

# 5c. Merge 4: put every announcement next to ALL the days of its stock.
pairs = events[["event_id", "permno", "n"]].merge(stock[["permno", "tday", "ar"]], on="permno", how="inner")

# 5d. Keep only the 3 days: day -1 (n - 1), day 0 (n) and day +1 (n + 1).
in_window = (pairs["tday"] >= pairs["n"] - 1) & (pairs["tday"] <= pairs["n"] + 1)
pairs = pairs[in_window]

# 5e. For each announcement, add up the abnormal returns ("sum") and count the days that have one ("count").
abn = pairs.groupby("event_id")["ar"].agg(["sum", "count"]).reset_index()
abn = abn.rename(columns={"sum": "car", "count": "days_found"})

# 5f. Put the result back on the announcements. how="left" keeps an announcement even if it has no day at all.
events = events.merge(abn, on="event_id", how="left")
events["days_found"] = events["days_found"].fillna(0)    # no day found = 0 days
r7, c7 = len(events), events["gvkey"].nunique()

print("how many of the 3 days were found, per announcement:")
events["days_found"].value_counts().sort_index()

how many of the 3 days were found, per announcement:


days_found
0.0       7
3.0    1768
Name: count, dtype: int64

<h2 style="background:#1f3a5f; color:white; padding:8px 14px; border-radius:4px">Step 6. The rules and the sample construction table</h2>

<div style="background:#e8f1fb; border-left:6px solid #2b6cb0; padding:8px 14px"><b>Rules.</b> An announcement stays in the final table only if (6) all 3 days (−1, 0, +1) have an abnormal return,
(4) last quarter's price is at least $5, and (5) the SUE is not extreme. We apply them one at a time and count after each one.</div>

In [14]:
# ===== Step 6a. all 3 daily abnormal returns found (class notebook; AI, see Appendix A; changed to daily) =====
events = events[events["days_found"] == 3]
r8, c8 = len(events), events["gvkey"].nunique()
print("announcements left:", r8)

announcements left: 1768


In [15]:
# ===== Step 6d. The sample construction table (class notebook; AI, see Appendix A; changed to quarterly) =====
# Every row is one step. r1..r10 and c1..c10 are the counts saved in the earlier steps.
sample_table = pd.DataFrame({
    "step": ["Compustat quarterly data, SIC " + SIC + " (fundq)",
             "Merge 1: after the link merge (with the link dates)",
             "Merge 2: after the merge with earlier quarters",
             "Rule 1: announcement date is known",
             "Rule 2: announcement date " + ANN_START + " to " + ANN_END,
             "Rule 3: SUE can be computed",
             "Rule 4: last quarter's price >= $" + str(MIN_PRC),
             "Rule 5: |SUE| <= " + str(MAX_SUE),
             "Merge 4: after adding the 3 daily returns",
             "Rule 6: all 3 daily abnormal returns found"],
    "rows left":      [r1, r2, r3, r4, r5, r6, r9, r10, r7, r8],
    "companies left": [c1, c2, c3, c4, c5, c6, c9, c10, c7, c8],
})

# Rows removed at each step = the difference from the row above. The first row removes nothing.
sample_table["rows removed"] = -sample_table["rows left"].diff()
sample_table["rows removed"] = sample_table["rows removed"].fillna(0).astype(int)

sample_table

,step,rows left,companies left,rows removed
0,"Compustat quarterly data, SIC 38 (fundq)",3865,177,0
1,Merge 1: after the link merge (with the link d...,3650,177,215
2,Merge 2: after the merge with earlier quarters,3650,177,0
3,Rule 1: announcement date is known,3624,177,26
4,Rule 2: announcement date 2021-01-01 to 2024-1...,2412,175,1212
5,Rule 3: SUE can be computed,2333,172,79
6,Rule 4: last quarter's price >= $5,1777,152,556
7,Rule 5: |SUE| <= 1,1775,151,2
8,Merge 4: after adding the 3 daily returns,1775,151,0
9,Rule 6: all 3 daily abnormal returns found,1768,151,7


In [16]:
# ===== Step 6e. The final table (class notebook; AI, see Appendix A; changed the columns) =====
print("Final table: one row =", "one earnings announcement (one company and one fiscal quarter)")
events[["sue", "car"]].describe()

Final table: one row = one earnings announcement (one company and one fiscal quarter)


,sue,car
count,1768.000000,1768.000000
mean,0.000153,-0.001883
std,0.045162,0.100863
min,-0.678407,-0.638571
25%,-0.003572,-0.051706
50%,0.000615,-0.004225
75%,0.004298,0.049295
max,0.841346,0.522563


In [17]:
# ===== Step 6f. Show the final table (AI, see Appendix A) =====
print("Final table: one row = one earnings announcement (one company and one fiscal quarter)")
print("rows:", len(events), "| companies:", events["gvkey"].nunique())
events[["gvkey", "fyearq", "fqtr", "ticker", "rdq", "day0", "sue", "car"]].head(10)

In [18]:
# ===== Close the WRDS connection (class notebook, unchanged) =====
conn.close()

<h2 style="background:#7b1e1e; color:white; padding:8px 14px; border-radius:4px">Step 7. Checks</h2>

In [19]:
# ===== Check 2: duplicates in the final table (AI, see Appendix A) =====
# duplicated() marks a row True if an earlier row has the same key. sum() counts the True values.
print("duplicate gvkey + datadate:", events.duplicated(["gvkey", "datadate"]).sum(), "(it must be 0)")
print("duplicate gvkey + fyearq + fqtr:", events.duplicated(["gvkey", "fyearq", "fqtr"]).sum(), "(it must be 0)")

duplicate gvkey + datadate: 0 (it must be 0)
duplicate gvkey + fyearq + fqtr: 0 (it must be 0)


In [20]:
# ===== Check 3: dates (AI, see Appendix A) =====

# (a) The announcement must come after the quarter ends. Count the rows where it does not.
too_early = events["rdq"] <= events["datadate"]
print("(a) announcements on or before the quarter end:", too_early.sum(), "(it should be 0)")

# (b) Day 0 must be a day the market was open, so every day0 must be a date in the market table.
is_trading_day = events["day0"].isin(mkt["date"])
print("(b) day 0 dates that are trading days:", is_trading_day.sum(), "out of", len(events))

# Example: announcements made on a weekend (dayofweek 5 = Saturday, 6 = Sunday) and the day 0 they got.
weekend = events[events["rdq"].dt.dayofweek >= 5]
print("\nannouncements made on a weekend:", len(weekend))
weekend[["ticker", "rdq", "day0"]].head()

In [21]:
# ===== Check 4: the 5 largest CARs (AI, see Appendix A) =====
# abs() removes the minus sign, so a -9% CAR and a +9% CAR count as equally big.
events["abs_car"] = events["car"].abs()
top5 = events.sort_values("abs_car", ascending=False).head(5)
top5[["ticker", "rdq", "epspxq", "eps_lag4", "prc_lag1", "sue", "car"]]

In [22]:
# ===== Check 5: Stryker's Announcements (AI, see Appendix A) =====
client = events[events["ticker"] == "SYK"]
print("SYK announcements in the final table:", len(client))
client[["fyearq", "fqtr", "datadate", "rdq", "day0", "epspxq", "eps_lag4", "prc_lag1", "sue", "car"]]

In [23]:
# ===== Check 6: the three days of Stryker's 2023 Q4 announcement (AI, see Appendix A) =====
row = client[client["rdq"] == "2024-01-30"].iloc[0]     # the 2023 Q4 row (announced 2024-01-30) in the Check 5 table
print("day 0:", row["day0"], "| SUE:", row["sue"], "| CAR:", row["car"])
three_days = stock[(stock["permno"] == row["permno"]) & (stock["tday"] >= row["n"] - 1) & (stock["tday"] <= row["n"] + 1)]
three_days[["date", "ret", "vwretd", "ar"]]

<h2 style="background:#7b1e1e; color:white; padding:8px 14px; border-radius:4px">Step 7. Results</h2>

In [24]:
# ===== Three SUE groups (AI, see Appendix A; changed after feedback: rank before qcut) =====
# Many announcements have the same SUE (for example exactly 0). qcut on SUE puts all tied values in one group,
# so the groups would not be the same size. rank(method="first") gives every announcement its own rank
# (ties are broken by the order of the rows), and qcut on the rank then makes 3 groups of (almost) the same size.
events["sue_rank"]  = events["sue"].rank(method="first")
events["sue_group"] = pd.qcut(events["sue_rank"], 3, labels=["Low", "Middle", "High"])
events.groupby("sue_group")["car"].agg(["count", "mean"])

,count,mean
sue_group,,
Low,589,-0.020058
Middle,589,-0.002844
High,590,0.017221


In [25]:
# ===== Question 1: Sample (AI, see Appendix A) =====
print("companies in each group:")
print(events.groupby("sue_group")["gvkey"].nunique())

# If the median is far from the mean, a few announcements pull the average.
print("\nmedian CAR in each group:")
print(events.groupby("sue_group")["car"].median())

# The companies with the most announcements in each group.
print("\nLow group:")
print(events[events["sue_group"] == "Low"]["ticker"].value_counts().head(3))
print("\nMiddle group:")
print(events[events["sue_group"] == "Middle"]["ticker"].value_counts().head(3))
print("\nHigh group:")
print(events[events["sue_group"] == "High"]["ticker"].value_counts().head(3))

In [26]:
# ===== Question 2: Extremes (AI, see Appendix A) =====
without_top5 = events.drop(top5.index)

print("before (all announcements):")
print(events.groupby("sue_group")["car"].agg(["count", "mean"]))
print("\nafter (without the 5 largest CARs):")
print(without_top5.groupby("sue_group")["car"].agg(["count", "mean"]))

before (all announcements):
           count      mean
sue_group                 
Low          589 -0.020058
Middle       589 -0.002844
High         590  0.017221

after (without the 5 largest CARs):
           count      mean
sue_group                 
Low          588 -0.019006
Middle       589 -0.002844
High         586  0.015557


In [27]:
# ===== Question 3: Stryker's Largest CAR (AI, see Appendix A) =====
client = events[events["ticker"] == "SYK"]
client.sort_values("abs_car", ascending=False).head(1)[["fyearq", "fqtr", "rdq", "day0", "sue", "car"]]